In [6]:
!pip install -q imagehash

In [7]:
import hashlib, re
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
from PIL import Image, ImageOps, UnidentifiedImageError
import imagehash

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SELECTED = {"corn", "tomato", "potato", "bell_pepper", "soybean"}

def find_dataset_root():
    for p in Path("/kaggle/input").rglob("*"):
        if p.is_dir() and (p / "train").exists() and (p / "val").exists():
            return p
    raise FileNotFoundError("Could not find a folder containing train/ and val/ under /kaggle/input")

def normalize_text(v):
    v = v.strip().lower().replace("_(maize)", "").replace("pepper,_bell", "bell_pepper")
    v = re.sub(r"[^a-z0-9]+", "_", v)
    return re.sub(r"_+", "_", v).strip("_")

def parse_class_name(name):
    crop_raw, disease_raw = name.split("___", 1) if "___" in name else (name, "unknown")
    crop, disease = normalize_text(crop_raw), normalize_text(disease_raw)
    return crop, disease, int(disease == "healthy")

def sha256_file(path, chunk=1024 * 1024):
    d = hashlib.sha256()
    with open(path, "rb") as f:
        for c in iter(lambda: f.read(chunk), b""):
            d.update(c)
    return d.hexdigest()

def compute_phash(path):
    try:
        with Image.open(path) as im:
            return str(imagehash.phash(ImageOps.exif_transpose(im).convert("RGB")))
    except Exception:
        return None

def read_metadata(path):
    try:
        with Image.open(path) as im:
            im = ImageOps.exif_transpose(im)
            return {"width": im.size[0], "height": im.size[1],
                    "channels": len(im.mode) if im.mode else None, "is_corrupt": 0}
    except Exception:
        return {"width": None, "height": None, "channels": None, "is_corrupt": 1}

def leaf_mask_hsv(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, np.array([20, 25, 20], np.uint8), np.array([100, 255, 255], np.uint8))
    v = hsv[:, :, 2]
    mask = cv2.bitwise_or(mask, (((v > 20) & (v < 230)).astype(np.uint8) * 255))
    k = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask, 8)
    if n <= 1:
        return mask
    largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    return np.where(labels == largest, 255, 0).astype(np.uint8)

def lesion_mask_hsv(img, leaf):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    lesion = cv2.inRange(hsv, np.array([5, 35, 20], np.uint8), np.array([35, 255, 220], np.uint8))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    lesion = cv2.bitwise_or(lesion, np.where(gray < 55, 255, 0).astype(np.uint8))
    lesion = cv2.bitwise_and(lesion, leaf)
    k = np.ones((3, 3), np.uint8)
    lesion = cv2.morphologyEx(lesion, cv2.MORPH_OPEN, k)
    return cv2.morphologyEx(lesion, cv2.MORPH_CLOSE, k)

def proxy_severity(path):
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:
        return {"leaf_area_px": None, "lesion_area_px": None,
                "severity_ratio": None, "severity_stage": "unreadable"}
    img = cv2.resize(img, (256, 256), interpolation=cv2.INTER_AREA)
    leaf = leaf_mask_hsv(img)
    lesion = lesion_mask_hsv(img, leaf)
    la, le = int(np.count_nonzero(leaf)), int(np.count_nonzero(lesion))
    if la == 0:
        return {"leaf_area_px": 0, "lesion_area_px": le,
                "severity_ratio": None, "severity_stage": "leaf_not_found"}
    r = le / la
    stage = "mild" if r < 0.05 else ("moderate" if r < 0.15 else "severe")
    return {"leaf_area_px": la, "lesion_area_px": le,
            "severity_ratio": round(float(r), 6), "severity_stage": stage}

def assign_duplicate_groups(df):
    df = df.copy()
    df["duplicate_group"] = ""
    gid = 0
    for _, g in df.dropna(subset=["sha256"]).groupby("sha256"):
        if len(g) > 1:
            gid += 1
            df.loc[g.index, "duplicate_group"] = f"exact_{gid:05d}"
    pid = 0
    for _, g in df.loc[df["duplicate_group"].eq("")].dropna(subset=["phash"]).groupby("phash"):
        if len(g) > 1:
            pid += 1
            df.loc[g.index, "duplicate_group"] = f"phash_{pid:05d}"
    return df

# ---------------- main run ----------------
DATASET_ROOT = find_dataset_root()
OUTPUT_DIR = Path("/kaggle/working")
print("Dataset root:", DATASET_ROOT)

records = []
for split in ["train", "val"]:
    for class_dir in sorted(p for p in (DATASET_ROOT / split).iterdir() if p.is_dir()):
        crop, disease, is_healthy = parse_class_name(class_dir.name)
        if crop not in SELECTED:
            continue
        for img_path in sorted(class_dir.rglob("*")):
            if not (img_path.is_file() and img_path.suffix.lower() in IMAGE_EXTENSIONS):
                continue
            rec = {
                "image_path": str(img_path),
                "relative_path": str(img_path.relative_to(DATASET_ROOT)),
                "split": split,
                "source_class_folder": class_dir.name,
                "crop": crop, "disease": disease,
                "label": f"{crop}__{disease}",
                "is_healthy": is_healthy,
                "fileThe most likely failure: the script uses `argparse` and `if __name__ == "__main__"`, which **breaks inside a Kaggle/Jupyter notebook cell** (argparse tries to parse the kernel's own arguments and raises `SystemExit: 2`). Also, your pasted message was cut off mid-code, so if you pasted the same truncated text into the cell, it would also fail with a syntax error.

The fix for notebooks is simple: skip argparse entirely and call `build_manifest()` directly.

## Notebook version — paste this as cells

**Cell 1 — install dependency:**

```python
!pip install -q imagehash
```

**Cell 2 — find the real dataset path first** (this is the #1 cause of failure — paths vary per Kaggle dataset copy):

```python
import os
for root, dirs, files in os.walk("/kaggle/input"):
    if dirs and "train" in dirs:
        print(root)
```

**Cell 3 — notebook-friendly manifest builder:**

```python
import hashlib, re
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
from PIL import Image, ImageOps, UnidentifiedImageError
import imagehash

DATASET_ROOT = Path("/kaggle/input/plantvillage/PlantVillage")  # adjust to Cell 2 output
OUTPUT_DIR = Path("/kaggle/working")
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SELECTED = {"corn", "tomato", "potato", "bell_pepper", "soybean"}


def normalize_text(value: str) -> str:
    value = value.strip().lower()
    value = value.replace("_(maize)", "").replace("pepper,_bell", "bell_pepper")
    value = re.sub(r"[^a-z0-9]+", "_", value)
    return re.sub(r"_+", "_", value).strip("_")


def parse_class_name(class_name: str):
    if "___" in class_name:
        crop_raw, disease_raw = class_name.split("___", 1)
    else:
        crop_raw, disease_raw = class_name, "unknown"
    crop, disease = normalize_text(crop_raw), normalize_text(disease_raw)
    return crop, disease, int(disease == "healthy")


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def compute_phash(path):
    try:
        with Image.open(path) as img:
            img = ImageOps.exif_transpose(img).convert("RGB")
            return str(imagehash.phash(img))
    except Exception:
        return None


def read_metadata(path):
    try:
        with Image.open(path) as img:
            img = ImageOps.exif_transpose(img)
            w, h = img.size
            return w, h, 0
    except Exception:
        return None, None, 1


def proxy_severity(path):
    img = cv2.imread(str(path))
    if img is None:
        return None, "unreadable"
    img = cv2.resize(img, (256, 256), interpolation=cv2.INTER_AREA)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    leaf = cv2.inRange(hsv, np.array(), np.array())[20][25][100][255]
    lesion = cv2.inRange(hsv, np.array(), np.array())[5][20][35][220][255]
    lesion = cv2.bitwise_and(lesion, leaf)
    leaf_area = int(np.count_nonzero(leaf))
    if leaf_area == 0:
        return None, "leaf_not_found"
    ratio = np.count_nonzero(lesion) / leaf_area
    stage = "mild" if ratio < 0.05 else ("moderate" if ratio < 0.15 else "severe")
    return round(float(ratio), 6), stage


records = []
for split in ["train", "val"]:
    split_dir = DATASET_ROOT / split
    assert split_dir.exists(), f"Missing: {split_dir} — check DATASET_ROOT from Cell 2"
    for class_dir in sorted(p for p in split_dir.iterdir() if p.is_dir()):
        crop, disease, is_healthy = parse_class_name(class_dir.name)
        if crop not in SELECTED:
            continue
        for img_path in sorted(class_dir.rglob("*")):
            if img_path.suffix.lower() not in IMAGE_EXTENSIONS:
                continue
            w, h, corrupt = read_metadata(img_path)
            ratio, stage = (None, "skipped") if corrupt else proxy_severity(img_path)
            records.append({
                "image_path": str(img_path),
                "split": split,
                "source_class_folder": class_dir.name,
                "crop": crop, "disease": disease,
                "label": f"{crop}__{disease}",
                "is_healthy": is_healthy,
                "width": w, "height": h, "is_corrupt": corrupt,
                "sha256": None if corrupt else sha256_file(img_path),
                "phash": None if corrupt else compute_phash(img_path),
                "severity_ratio": ratio, "severity_stage": stage,
            })

manifest = pd.DataFrame(records)
manifest.to_csv(OUTPUT_DIR / "plantvillage_selected_manifest.csv", index=False)

print(f"Rows: {len(manifest)} | Classes: {manifest['label'].nunique()} | Corrupt: {manifest['is_corrupt'].sum()}")
display(manifest.groupby(["split", "label"]).size().unstack(fill_value=0))
```

## Two things to tell me if it still fails

1. The **exact error message** (last lines of the traceback).
2. The output of **Cell 2** — so I can correct the dataset path.

Once the manifest CSV exists in `/kaggle/working`, the next step is the duplicate/leakage check across train and val.

SyntaxError: unterminated string literal (detected at line 129) (729089299.py, line 129)